# Minimal Dynesty.jl Corner Example

This notebook runs a two-parameter Gaussian posterior, saves a corner plot, and includes a threaded sampler cell. For real threaded execution, start the Julia kernel with multiple threads, for example `JULIA_NUM_THREADS=4` or `julia --threads=4`.


In [ ]:
using Dynesty
using Random

PLOTS_PKGID = Base.PkgId(Base.UUID("91a5bcdd-55d7-5caf-9e0b-520d859cae80"), "Plots")

function load_plots()
    try
        return Base.require(PLOTS_PKGID)
    catch err
        if !(err isa ArgumentError) && !occursin("Package Plots", sprint(showerror, err))
            rethrow()
        end
        error(
            "Plots.jl is required to render the corner plot. Run " *
            "`julia --project=examples -e 'using Pkg; Pkg.instantiate()'` first.",
        )
    end
end

TRUE_MEAN = [0.25, -0.35]
TRUE_SIGMA = [0.12, 0.20]


In [ ]:
prior_transform(u) = [-1.0 + 2.0 * u[1], -1.0 + 2.0 * u[2]]
loglikelihood(theta) = -0.5 * sum(((theta .- TRUE_MEAN) ./ TRUE_SIGMA) .^ 2)

function run_sampler(;
    rng=MersenneTwister(42),
    nlive=150,
    dlogz=0.05,
    parallel=:serial,
    queue_size=nothing,
    proposal_scheduler=:batch,
)
    sampler = NestedSampler(
        loglikelihood,
        prior_transform,
        2;
        nlive,
        bound=:multi,
        sample=:unif,
        rng,
        parallel,
        queue_size,
        proposal_scheduler,
        enlarge=1.1,
        bootstrap=0,
    )
    run_nested!(sampler; dlogz, print_progress=false)
    return (; sampler, res=results(sampler))
end

backend_name(sampler) = string(nameof(typeof(sampler.map_backend)))

function posterior_summary(fit)
    res = fit.res
    sampler = fit.sampler
    weights = importance_weights(res)
    mean = vec(sum(res.samples .* weights; dims=1))
    equal_weight_samples = samples_equal(res; rng=MersenneTwister(1234))
    return (
        logz=res.logz[end],
        logzerr=res.logzerr[end],
        nsamples=length(res.logl),
        mean=mean,
        equal_weight_samples=equal_weight_samples,
        backend=backend_name(sampler),
        queue_size=sampler.map_backend.queue_size,
        proposal_scheduler=sampler.proposal_scheduler,
        proposal_tasks_submitted=sampler.proposal_tasks_submitted,
        proposal_batches_submitted=sampler.proposal_batches_submitted,
        threads=Threads.nthreads(),
    )
end

function corner_figure(res; size=(720, 720))
    plots = load_plots()
    return Base.invokelatest(
        plots.plot,
        cornerplot(
            res;
            dims=[1, 2],
            labels=["theta1", "theta2"],
            truths=TRUE_MEAN,
            smooth=[30, 30],
        );
        size,
    )
end

function save_corner_plot(res; path=joinpath(@__DIR__, "output", "minimal_corner.png"))
    plots = load_plots()
    mkpath(dirname(path))
    fig = corner_figure(res)
    Base.invokelatest(plots.savefig, fig, path)
    return path
end

function main(;
    make_plot=false,
    plot_path=joinpath(@__DIR__, "output", "minimal_corner.png"),
    parallel=:serial,
    queue_size=nothing,
    proposal_scheduler=:batch,
    nlive=150,
    dlogz=0.05,
    seed=42,
)
    fit = run_sampler(;
        rng=MersenneTwister(Int(seed)),
        nlive=Int(nlive),
        dlogz=Float64(dlogz),
        parallel,
        queue_size,
        proposal_scheduler,
    )
    summary = posterior_summary(fit)
    saved_plot = make_plot ? save_corner_plot(fit.res; path=plot_path) : nothing
    return merge(summary, (plot=saved_plot, results=fit.res))
end


In [ ]:
serial_summary = main(; make_plot=true, nlive=1024);
(
    logz=serial_summary.logz,
    logzerr=serial_summary.logzerr,
    nsamples=serial_summary.nsamples,
    mean=serial_summary.mean,
    backend=serial_summary.backend,
    queue_size=serial_summary.queue_size,
    proposal_tasks=serial_summary.proposal_tasks_submitted,
    plot=serial_summary.plot,
)


In [ ]:
queue_size = min(4, Threads.nthreads())

parallel_summary = if queue_size < 2
    @warn "Restart the Julia notebook kernel with multiple threads to run this cell in parallel." threads=Threads.nthreads()
    nothing
else
    main(;
        make_plot=true,
        nlive=1024,
        parallel=:threads,
        queue_size,
        proposal_scheduler=:batch,
        plot_path=joinpath(@__DIR__, "output", "minimal_corner_parallel.png"),
    )
end;

isnothing(parallel_summary) ? nothing : (
    logz=parallel_summary.logz,
    logzerr=parallel_summary.logzerr,
    nsamples=parallel_summary.nsamples,
    mean=parallel_summary.mean,
    backend=parallel_summary.backend,
    threads=parallel_summary.threads,
    queue_size=parallel_summary.queue_size,
    proposal_tasks=parallel_summary.proposal_tasks_submitted,
    proposal_batches=parallel_summary.proposal_batches_submitted,
    plot=parallel_summary.plot,
)


In [ ]:
# Display an inline corner plot from the threaded run when available;
# otherwise display the serial run's posterior.
plot_results = isnothing(parallel_summary) ? serial_summary.results : parallel_summary.results
corner_figure(plot_results)
